# Improve Parking Counting

This notebook provides scaffolding for training and evaluating your own
parking detection model.  The goal: **beat the Grounding DINO zero-shot baseline**.

## 1. Datasets

Download one (or more) of these datasets:

| Dataset | Description | Link |
|---------|-------------|------|
| **ParkSeg12k** | 12k annotated parking lot images with segmentation masks | [GitHub](https://github.com/geohai/ParkSeg12k) |
| **APKLOT** | Aerial parking lot dataset with bounding boxes | [Kaggle](https://www.kaggle.com/datasets) |
| **SpaceNet** | Satellite imagery with building footprints | [spacenet.ai](https://spacenet.ai/) |
| **NAIP** | 1m aerial imagery from USDA | [NAIP](https://naip-usdaonline.hub.arcgis.com/) |

See `../data/README.md` for more details.

In [ ]:
# Uncomment and set your dataset path:
# DATASET_DIR = "../data/parkseg12k"

## 2. Load dataset

In [ ]:
# TODO: Load your training images and annotations here
#
# Example for ParkSeg12k:
#   from pathlib import Path
#   images = sorted(Path(DATASET_DIR, "images").glob("*.png"))
#   masks  = sorted(Path(DATASET_DIR, "masks").glob("*.png"))
#   print(f"Loaded {len(images)} images")

## 3. Define your model

In [ ]:
# TODO: Define or load your model here
#
# Ideas:
#   - Fine-tune YOLOv8: `from ultralytics import YOLO; model = YOLO('yolov8n.pt')`
#   - Fine-tune Grounding DINO on parking data
#   - Train a U-Net for segmentation
#   - Use SAM with custom prompts

## 4. Train

In [ ]:
# TODO: Training loop here
#
# Example (YOLOv8):
#   model.train(data="parking.yaml", epochs=50, imgsz=640)

## 5. Evaluate

In [ ]:
# TODO: Run your model on test images and compute metrics

## 6. Compare against baselines

Use this cell to compare your model against the ParkSight baselines.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from parksight import fetch, count
from parksight.detect import ParkingDetector

# Test address
ADDRESS = "Georgia Tech, Atlanta, GA"
gdf, (lat, lon) = fetch.get_parking_data(ADDRESS, dist=300)
gdf_3857 = gdf.to_crs(epsg=3857)
polygons = gdf_3857[gdf_3857.geometry.geom_type == "Polygon"]

if len(polygons) > 0:
    geom = polygons.iloc[0].geometry
    tile = fetch.get_satellite_tile(geom)

    cv_count = count.count_edges(geom)

    detector = ParkingDetector()
    ml_count = detector.count_spots(tile)

    # TODO: Replace with your model's count
    your_count = 0  # your_model.predict(tile)

    print(f"CV baseline:    {cv_count}")
    print(f"ML baseline:    {ml_count}")
    print(f"Your model:     {your_count}")
    print(f"Delta vs ML:    {your_count - ml_count:+d}")